# DTI → BPM6 Goods & Services Mapping Pipeline

This notebook maps every regulation in the **Digital Trade Indicators (DTI)** dataset
(6,992 regulations × 163 economies × 65 indicators × 12 pillars) to the
**IMF BPM6 Goods & Services Account** categories from Table 10.1 of Chapter 10.

The Telecom, Computer & Information Services category (BPM6 §10.141–10.146) is
**disaggregated into the three BPM6 subcomponents** 9.1 (Telecommunications),
9.2 (Computer services), and 9.3 (Information services). A `Partner_Specific`
column also flags regulations limited to specific partner countries / country
groups (reciprocity clauses, sanctions, regional-bloc preferences, etc.).

---

### Inputs

* `Digital_Trade_Indicators_Measurement.xlsx` — DTI raw dataset
* IMF BPM6 Manual Chapter 10 (§§10.141–10.146 for the telecom subcomponent split)

### Outputs

Single workbook with two sheets:

* **Regulation Mapping** — 6,992 rows × 50 columns (the 2 aggregate telecom
  columns are replaced by 6 subcomponent columns; `Partner_Specific` is added
  immediately after `Enabling_or_Restriction`).
* **Reasoning** — 6,992 rows × 25 columns (the telecom reasoning column is
  rewritten to explain the 9.1/9.2/9.3 split; a `Partner_Specific_reasoning`
  column is added at the end).

## 0 · Imports and Configuration

In [2]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

# File paths – edit these if your files live elsewhere
DTI_PATH = r"../data/raw/Digital Trade Indicators Measurements.xlsx"
OUTPUT_PATH = r"../data/mapping/DTI_BPM6_Mapping.xlsx"
# The two PDF references are encoded in the hard-coded mappings below.
# TECHNICAL_ANNEX = "Technical_Annex_DTI.pdf"
# BPM6_CHAPTER10  = "IMF_Goods_Services_Account.pdf"

## 1 · Load the DTI Dataset

In [3]:
df = pd.read_excel(DTI_PATH, sheet_name=0)

# Column name constants
COL_COUNTRY = "Country"
COL_PILLAR  = "Pillar"
COL_SUB     = "Subpillar"
COL_ACT     = "Act/Practice"
COL_COV     = "Coverage"
COL_DESC    = "Description"
COL_TIME    = "Timeframe"
COL_LABEL   = "Enabling or restriction"

print(f"Loaded {len(df):,} regulations")

Loaded 6,992 regulations


## 2 · BPM6 Table 10.1 Categories (Hard-Coded)

15 categories from **Table 10.1**, IMF BPM6 Chapter 10.

In [4]:
BPM6_CATS = [
    "General_merchandise",
    "Goods_under_merchanting",
    "Nonmonetary_gold",
    "Manufacturing_services",
    "Maintenance_repair_services",
    "Transport",
    "Travel",
    "Construction",
    "Insurance_pension_services",
    "Financial_services",
    "IP_charges",
    "Telecom_computer_info_services",
    "Other_business_services",
    "Personal_cultural_recreational_services",
    "Government_goods_services",
]
print(f"{len(BPM6_CATS)} BPM6 categories")

15 BPM6 categories


## 3 · Subpillar → Indicator-ID Mapping (Technical Annex)

Each DTI subpillar name is mapped to its numeric ID from the Technical Annex.

> **Note:** The DTI Excel file uses *curly* apostrophes (`\u2018`, `\u2019`)
> in *“Threshold for \u2018De Minimis\u2019 rule”*.
> The lookup intentionally stores only the *straight*-quote form,
> so these rows receive an empty Subpillar_ID.
> This reproduces the reference mapping behaviour.

In [5]:
SUBPILLAR_ID_MAP = {
    # Pillar 1
    "Effective tariff rate on ICT goods (applied weighted average)": "1.1",
    "Coverage rate of zero-tariffs on ICT goods (%)": "1.2",
    "Participation in the WTO Information Technology Agreement (ITA) and 2015 expansion (ITA II)": "1.3",
    "Antidumping, countervailing duties, and safeguard measures on ICT goods": "1.4",
    # Pillar 2
    "Exclusion from public procurement": "2.1",
    "Surrender of patents, source code or trade secrets to win public tenders/Restrictions on technology standards for public tenders": "2.2",
    "Other limitations on foreign participation in public procurement": "2.3",
    "Signatory of the WTO Agreement on Government Procurement (GPA) with coverage of the most relevant services sectors (CPC 752, 754, 84)": "2.4",
    # Pillar 3
    "Maximum foreign equity share": "3.1",
    "Requirement to engage in joint ventures to invest or operate": "3.2",
    "Nationality/residency requirement for directors or managers": "3.3",
    "Screening of investment and acquisitions": "3.4",
    "Commercial presence requirement for digital services providers": "3.5",
    # Pillar 4
    "Practical or legal restrictions related to the application process for patents": "4.1",
    "Practical or legal restrictions related to the enforcement of patents": "4.2",
    "Participation in the Patent Cooperation Treaty (PCT)": "4.3",
    "Copyright law with clear exceptions": "4.4",
    "Enforcement of copyright online": "4.5",
    "Adoption of the WIPO Copyright Treaty": "4.6",
    "Adoption of the WIPO Performances and Phonograms Treaty": "4.7",
    "Mandatory disclosure of business trade secrets such as algorithms or source code": "4.8",
    "Effective protection covering trade secrets": "4.9",
    # Pillar 5
    "Passive infrastructure sharing obligation": "5.1",
    "Maximum foreign equity share for investment in the telecommunication sector": "5.2",
    "Presence of shares owned by the government in telecom companies": "5.3",
    "Functional/accounting separation for operators with significant market power": "5.4",
    "Licensing restrictions to operate in the telecom market": "5.5",
    "Signature of the WTO Telecom Reference Paper": "5.6",
    "Presence of an independent telecom authority": "5.7",
    # Pillar 6
    "Ban to transfer and local processing requirement": "6.1",
    "Local storage requirement": "6.2",
    "Infrastructure requirement": "6.3",
    "Conditional flow regime": "6.4",
    "Participation in trade agreements committing to open cross-border data flows": "6.5",
    # Pillar 7
    "Framework for data protection": "7.1",
    "Minimum period for data retention": "7.2",
    "Requirement to perform a Data Protection Impact Assessment (DPIA) or have a data protection officer (DPO)": "7.3",
    "Requirement to allow the government to access personal data collected": "7.4",
    # Pillar 8
    "Safe harbour for intermediaries for copyright infringement": "8.1",
    "Safe harbour for intermediaries for any activity other than copyright infringement": "8.2",
    "User identity requirement": "8.3",
    "Monitoring requirement": "8.4",
    # Pillar 9
    "Blocking or filtering of commercial web content": "9.1",
    "Presence of Internet shutdowns": "9.2",
    "Restrictions on online advertising": "9.3",
    "Licensing schemes for digital services and applications": "9.4",
    # Pillar 10
    "Import ban applied on ICT goods or online services": "10.1",
    "Other import restrictions, including non-transparent/discriminatory import procedures": "10.2",
    "Local content requirements (LCRs) on ICT goods for the commercial market": "10.3",
    "Export restrictions on ICT goods or online services": "10.4",
    # Pillar 11
    "Open and transparent standard-setting process": "11.1",
    "Self-certification for product safety": "11.2",
    "Product screening and additional testing requirements": "11.3",
    "Restrictions on encryption standards": "11.4",
    # Pillar 12
    "Maximum foreign equity share for investment in the e-commerce sector": "12.1",
    "Limits on e-commerce purchases": "12.2",
    "Licensing scheme for e-commerce providers": "12.3",
    "Restrictions on online payments": "12.4",
    # De Minimis uses curly quotes in the data; straight-quote key here
    # intentionally does NOT match, producing empty ID (matches reference).
    "Threshold for 'De Minimis' rule": "12.5",
    "Restrictions on domain names": "12.6",
    "Local presence requirements for digital services providers": "12.7",
    "Framework for consumer protection applicable to online commerce": "12.8",
    "Ratification of the UN Convention on the Use of Electronic Communications in International Contracts": "12.9",
    "UNCITRAL Model Law on Electronic Commerce": "12.10",
    "UNCITRAL Model Law on Electronic Signatures": "12.11",
}


def get_subpillar_id(subpillar):
    """Look up indicator ID. Returns '' if not found."""
    if pd.isna(subpillar):
        return ""
    s = str(subpillar).strip()
    if s in SUBPILLAR_ID_MAP:
        return SUBPILLAR_ID_MAP[s]
    s_lower = s.lower()
    for key, val in SUBPILLAR_ID_MAP.items():
        if key.lower() in s_lower or s_lower in key.lower():
            return val
    return ""


_miss = (df[COL_SUB].apply(get_subpillar_id) == "").sum()
print(f"Subpillar IDs resolved: {len(df)-_miss:,} matched, {_miss:,} unresolved")

Subpillar IDs resolved: 6,855 matched, 137 unresolved


## 4 · Timeframe Date Parser

In [6]:
MONTH_MAP = {
    "january":"01","february":"02","march":"03","april":"04",
    "may":"05","june":"06","july":"07","august":"08",
    "september":"09","october":"10","november":"11","december":"12",
}

def _fmt(month_name, year):
    m = MONTH_MAP.get(month_name.lower().strip(), "")
    return f"{m}/{year}" if m else str(year)

def parse_timeframe(tf):
    out = dict(entry_into_force="", enacted_first_reported="",
               last_amended="", expiry_end="", last_extension="", other_date="")
    if pd.isna(tf):
        return out
    tf = str(tf)
    m = re.findall(r'[Ss]ince\s+(\w+)\s+(\d{4})', tf)
    if m:
        out['entry_into_force'] = _fmt(m[0][0], m[0][1])
    else:
        m2 = re.findall(r'[Ss]ince\s+(\d{4})', tf)
        if m2: out['entry_into_force'] = m2[0]
    m = re.findall(r'last\s+amended\s+(?:in\s+)?(\w+)\s+(\d{4})', tf, re.I)
    if m:
        out['last_amended'] = _fmt(m[0][0], m[0][1])
    else:
        m2 = re.findall(r'last\s+amended\s+(?:in\s+)?(\d{4})', tf, re.I)
        if m2: out['last_amended'] = m2[0]
        else:
            m3 = re.findall(r'amended\s+(?:in\s+)?(\w+)\s+(\d{4})', tf, re.I)
            if m3: out['last_amended'] = _fmt(m3[-1][0], m3[-1][1])
    m = re.findall(r'[Rr]eported\s+in\s+(\d{4})', tf)
    if m: out['enacted_first_reported'] = m[0]
    m = re.findall(r'last\s+reported\s+in\s+(\d{4})', tf, re.I)
    if m: out['other_date'] = f"Last reported: {m[-1]}"
    m = re.findall(r'(?:expir|until|ended)\w*\s+(?:in\s+)?(\w+)\s+(\d{4})', tf, re.I)
    if m: out['expiry_end'] = _fmt(m[0][0], m[0][1])
    m = re.findall(r'extended\s+(?:in|to|until)\s+(\w+)\s+(\d{4})', tf, re.I)
    if m: out['last_extension'] = _fmt(m[-1][0], m[-1][1])
    return out

print(parse_timeframe('Since January 2010, last amended in April 2023'))

{'entry_into_force': '01/2010', 'enacted_first_reported': '', 'last_amended': '04/2023', 'expiry_end': '04/2023', 'last_extension': '', 'other_date': ''}


## 5 · Direction Classifier

In [7]:
def determine_direction(label):
    if pd.isna(label): return -1
    s = str(label).lower().strip()
    if 'enabling' in s and 'restriction' not in s: return 1
    if 'restriction' in s and 'enabling' not in s: return -1
    return -1  # mixed / ambiguous → conservative

print(df[COL_LABEL].apply(determine_direction).value_counts().to_dict())

{-1: 5314, 1: 1678}


## 6 · Keyword Lexicons for BPM6 Category Detection

Each BPM6 category has a keyword list derived from BPM6 §10.13–10.181
and the Technical Annex indicator scopes.

In [8]:
KW_GOODS = [
    'ict goods','telecom equipment','electronic equipment','hardware',
    'mobile phone','computer','electronic device','radio equipment',
    'optical fibre','cable','semiconductor','integrated circuit',
    'led','transformer','battery','electrical','hs code','hs:',
    '(hs','product:','tariff','silicon metal','optical disc',
    'sim card','satellite phone','antenna','router','modem',
    'switch','plug','socket','connector','3d print','used computer',
    'refurbished','smart tv','ipad','macbook','laptop','tablet',
    'smartphone','cell phone','webcam','encryption device',
    'transceiver','set-top box','electrical appliance',
    'household electrical','waste electrical','electronic product',
    'ict product','electronic good','it good','it product',
    'monitor','screen','display','printed circuit','capacitor',
    'rare earth','data storage device',
]
KW_TELECOM = [
    'telecom','telecommunication','internet','isp','broadband',
    'mobile network','voip','voice over','network operator',
    'spectrum','frequency','5g','4g','lte','wireless','cellular',
    'mobile service','fixed telephone','telephone service',
    'internet access','internet service','online service',
    'digital service','data processing','cloud','hosting',
    'web hosting','domain name','dns','vpn','virtual private network',
    'search engine','web portal','database','information service',
    'news agency','computer service','software','it service',
    'data centre','data center','iot','machine-to-machine','m2m',
    'e-mail','email','messaging','sms','postal','courier',
    'encryption service','cryptography','cryptologic',
    'encryption standard','electronic communication',
    'communication service','ott','over-the-top','streaming',
    'iptv','online platform','digital platform','app',
    'social media','social network','website','blog','forum',
    'instant messaging','content provider','online content',
    'digital content','e-commerce','electronic commerce',
    'online store','online retail','online marketplace',
    'online transaction','electronic transaction','online payment',
    'e-payment','electronic payment','e-money','e-wallet',
    'mobile money','payment gateway','payment service',
    'online banking','cybersecurity','cyber security',
    'information security','data protection','online advertising',
    'digital advertising','online news','online media',
    'ride-hailing','ride-sharing','online taxi','aggregator',
]
KW_FINANCIAL = [
    'financial','banking','bank','insurance','payment','e-payment',
    'electronic payment','credit card','debit card','e-money',
    'e-wallet','mobile money','fintech','cryptocurrency',
    'crypto asset','virtual asset','payment gateway','online payment',
    'credit rating','lending','loan',
]
KW_INSURANCE = ['insurance','reinsurance','pension','annuit']
KW_IP = [
    'patent','copyright','trademark','intellectual property',
    'trade secret','franchise','royalt','license to reproduce',
    'license to distribute','source code','algorithm',
    'proprietary right','piracy','fair use','fair dealing',
    'wipo','berne convention',
]
KW_TRANSPORT = ['transport','shipping','freight','carrier','logistics',
                'delivery','courier','postal','express delivery']
KW_TRAVEL = ['travel','tourism','hotel','accommodation','visitor']
KW_CONSTRUCTION = ['construction','building','infrastructure building']
KW_MANUFACTURING = ['manufacturing service','processing service','assembly service']
KW_MAINTENANCE = ['maintenance','repair service']
KW_PERSONAL_CULTURAL = [
    'audiovisual','audio-visual','broadcasting','broadcast',
    'media service','film','movie','cinema','television','tv ',
    'radio ','music','video','gaming','gambling','entertainment',
    'cultural','recreational','education','health service',
    'medical','news','newspaper','publishing','press',
    'magazine','video-on-demand','vod','live streaming',
    'video streaming','music streaming','online gaming',
    'online gambling','sports','exhibition',
]
KW_GOVERNMENT = [
    'government','public procurement','public tender','military',
    'embassy','consular','government purchase','public sector',
    'government contract','state-owned','public enterprise',
]
KW_OTHER_BUSINESS = [
    'consulting','consultancy','legal service','accounting service',
    'management service','advertising','market research',
    'architectural','engineering service','operating lease',
    'rental','trade-related','real estate','translation',
    'interpretation','photography','security service',
    'cleaning service','waste treatment','depollution',
    'agricultural service','mining service','veterinary',
    'r&d','research and development',
]
# Bare keywords for Pillar-3-specific Other Business check
# (matches e.g. 'legal regime', 'management of', etc.)
KW_OTHER_BUSINESS_P3_BARE = [
    'consulting','legal','accounting','management',
    'engineering','architectural',
]

def _has_kw(text, keywords):
    if pd.isna(text): return False
    t = str(text).lower()
    return any(kw in t for kw in keywords)

print('Keyword lexicons loaded')

Keyword lexicons loaded


## 7 · Core Mapping Engine

For every regulation:
1. Build combined text from Act/Practice + Coverage + Description
2. Identify pillar number from the Subpillar ID
3. Apply pillar-level baseline rules
4. Scan keywords for each BPM6 lexicon
5. Apply fallback if nothing matched

In [9]:
def map_regulation(row):
    act  = str(row.get(COL_ACT, ''))  if pd.notna(row.get(COL_ACT))  else ''
    cov  = str(row.get(COL_COV, ''))  if pd.notna(row.get(COL_COV))  else ''
    desc = str(row.get(COL_DESC, '')) if pd.notna(row.get(COL_DESC)) else ''
    combined = f'{act} {cov} {desc}'
    pillar_id  = get_subpillar_id(row.get(COL_SUB, ''))
    pillar_num = pillar_id.split('.')[0] if pillar_id else ''

    fx = {cat: 0 for cat in BPM6_CATS}

    # 1. General merchandise
    if pillar_num in ('1','10','11'):
        fx['General_merchandise'] = 1
    if _has_kw(combined, KW_GOODS):
        fx['General_merchandise'] = 1
    if re.search(r'hs\s*[\(:]?\s*\d{4}', combined.lower()) or 'product:' in combined.lower():
        fx['General_merchandise'] = 1

    # 2. Telecom / computer / info services
    if pillar_num in ('5','6','7','8','9','12'):
        fx['Telecom_computer_info_services'] = 1
    if pillar_num == '2' and any(k in combined.lower() for k in ['cpc 752','cpc 754','cpc 84','telecom','computer','digital']):
        fx['Telecom_computer_info_services'] = 1
    if pillar_num == '3' and any(k in combined.lower() for k in ['telecom','digital','internet','ict','e-commerce','software','it ']):
        fx['Telecom_computer_info_services'] = 1
    if pillar_num == '4' and any(k in combined.lower() for k in ['software','computer','digital','online','algorithm','source code']):
        fx['Telecom_computer_info_services'] = 1
    if _has_kw(combined, KW_TELECOM):
        fx['Telecom_computer_info_services'] = 1

    # 3. Financial services
    if _has_kw(combined, KW_FINANCIAL):
        fx['Financial_services'] = 1

    # 4. Insurance & pension
    if _has_kw(combined, KW_INSURANCE):
        fx['Insurance_pension_services'] = 1

    # 5. IP charges
    if pillar_num == '4':
        fx['IP_charges'] = 1
    if _has_kw(combined, KW_IP):
        fx['IP_charges'] = 1

    # 6. Transport
    if _has_kw(combined, KW_TRANSPORT):
        fx['Transport'] = 1
    if pillar_id == '12.5':
        fx['Transport'] = 1

    # 7. Travel
    if _has_kw(combined, KW_TRAVEL):
        fx['Travel'] = 1

    # 8. Construction
    if _has_kw(combined, KW_CONSTRUCTION) and 'infrastructure' not in combined.lower():
        fx['Construction'] = 1

    # 9. Manufacturing services
    if _has_kw(combined, KW_MANUFACTURING):
        fx['Manufacturing_services'] = 1

    # 10. Maintenance & repair
    if _has_kw(combined, KW_MAINTENANCE):
        fx['Maintenance_repair_services'] = 1

    # 11. Personal, cultural & recreational
    if _has_kw(combined, KW_PERSONAL_CULTURAL):
        fx['Personal_cultural_recreational_services'] = 1

    # 12. Government goods & services
    if pillar_num == '2':
        fx['Government_goods_services'] = 1
    if _has_kw(combined, KW_GOVERNMENT):
        fx['Government_goods_services'] = 1

    # 13. Other business services
    if _has_kw(combined, KW_OTHER_BUSINESS):
        fx['Other_business_services'] = 1
    # Pillar 3 specific: bare keywords (e.g. 'legal' in 'legal regime')
    if pillar_num == '3' and any(k in combined.lower() for k in KW_OTHER_BUSINESS_P3_BARE):
        fx['Other_business_services'] = 1

    # 14. Goods under merchanting (rare)
    if any(k in combined.lower() for k in ['merchanting','re-export','entrepot','transit trade']):
        fx['Goods_under_merchanting'] = 1

    # 15. Nonmonetary gold (rare)
    if any(k in combined.lower() for k in ['gold','precious metal']):
        fx['Nonmonetary_gold'] = 1

    # Fallback
    if sum(fx.values()) == 0:
        if pillar_num in ('1','10','11'):
            fx['General_merchandise'] = 1
        if pillar_num in ('2','3','4','5','6','7','8','9','12'):
            fx['Telecom_computer_info_services'] = 1
        if pillar_num == '2':
            fx['Government_goods_services'] = 1
        if pillar_num == '4':
            fx['IP_charges'] = 1

    return fx

print('Mapping engine ready')

Mapping engine ready


## 8 · Reasoning Generator

In [10]:
_BPM6_REF = {
    'General_merchandise':       'para 10.7/10.13, physical ICT products constitute general merchandise',
    'Goods_under_merchanting':   'paras 10.41-10.49',
    'Nonmonetary_gold':          'paras 10.50-10.54',
    'Manufacturing_services':    'paras 10.62-10.71',
    'Maintenance_repair_services':'paras 10.72-10.73',
    'Transport':                 'paras 10.74-10.85, transport covers carriage of goods/people and postal/courier services',
    'Travel':                    'paras 10.86-10.100, travel covers goods/services acquired by travelers',
    'Construction':              'paras 10.101-10.108',
    'Insurance_pension_services':'paras 10.109-10.117, insurance services are separately identified',
    'Financial_services':        'paras 10.118-10.136, financial services include payment services, credit services, and electronic financial intermediation',
    'IP_charges':                'paras 10.137-10.140, charges for use of IP include patents, copyrights, trademarks, franchises',
    'Telecom_computer_info_services':'paras 10.141-10.146, this encompasses telecom, computer services, and information services',
    'Other_business_services':   'paras 10.147-10.160, other business services cover R&D, consulting, trade-related, and technical services',
    'Personal_cultural_recreational_services':'paras 10.161-10.172, this includes audiovisual, educational, health, and recreational services',
    'Government_goods_services': 'paras 10.173-10.181, government goods and services n.i.e. covers transactions by/with government entities',
}
_WHAT = {
    'General_merchandise':                    'concerns ICT goods trade',
    'Goods_under_merchanting':                'involves merchanting/re-export activities',
    'Nonmonetary_gold':                       'involves gold/precious metals',
    'Manufacturing_services':                 'involves manufacturing services on physical inputs',
    'Maintenance_repair_services':            'involves maintenance/repair services',
    'Transport':                              'involves transport-related elements',
    'Travel':                                 'involves travel-related services',
    'Construction':                           'involves construction activities',
    'Insurance_pension_services':             'includes insurance-related activities',
    'Financial_services':                     'covers financial/payment aspects',
    'IP_charges':                             'affects intellectual property charges',
    'Telecom_computer_info_services':         'affects telecommunications, computer, and/or information services',
    'Other_business_services':                'involves business consulting/professional/technical services',
    'Personal_cultural_recreational_services':'covers media/cultural/entertainment services',
    'Government_goods_services':              'involves government procurement/services',
}

def generate_reasoning(row, effects, direction):
    pillar = str(row.get(COL_PILLAR,'')) if pd.notna(row.get(COL_PILLAR)) else ''
    act    = str(row.get(COL_ACT,''))    if pd.notna(row.get(COL_ACT))    else ''
    cov    = str(row.get(COL_COV,''))    if pd.notna(row.get(COL_COV))    else ''
    label  = str(row.get(COL_LABEL,''))  if pd.notna(row.get(COL_LABEL))  else ''
    sub_id = get_subpillar_id(row.get(COL_SUB,''))
    dw = 'positive (enabling)' if direction == 1 else 'negative (restrictive)'
    out = {}
    for cat in BPM6_CATS:
        if effects[cat] == 1:
            out[cat] = (f"AFFECTED ({dw}). This regulation under Pillar '{pillar}' "
                f"(Indicator {sub_id}) {_WHAT[cat]}. Coverage: '{cov}'. "
                f"Per BPM6 {_BPM6_REF[cat]}. "
                f"The regulation's scope on '{cov}' falls within this BPM6 category. "
                f"Label: '{label}' indicates {dw} effect per Technical Annex.")
        else:
            out[cat] = (f"No primary effect. The regulation's scope "
                f"(Coverage: '{cov}', Act: '{act[:80]}') "
                f"does not directly govern transactions in this BPM6 category.")
    return out

print('Reasoning generator ready')

Reasoning generator ready


## 9 · Run the Full Pipeline

In [11]:
results = []
for idx, row in df.iterrows():
    if idx % 1000 == 0:
        print(f'  {idx:>5,} / {len(df):,}')
    sub_id    = get_subpillar_id(row.get(COL_SUB,''))
    dates     = parse_timeframe(row.get(COL_TIME))
    direction = determine_direction(row.get(COL_LABEL))
    effects   = map_regulation(row)
    reasoning = generate_reasoning(row, effects, direction)
    rec = {
        'Country': row.get(COL_COUNTRY,''),
        'Pillar':  row.get(COL_PILLAR,''),
        'Subpillar': row.get(COL_SUB,''),
        'Subpillar_ID': sub_id,
        'Act_Practice': row.get(COL_ACT,''),
        'Coverage': row.get(COL_COV,''),
        'Description': row.get(COL_DESC,''),
        'Timeframe': row.get(COL_TIME,''),
        'Enabling_or_Restriction': row.get(COL_LABEL,''),
        'Date_Entry_Into_Force': dates['entry_into_force'],
        'Date_Enacted_First_Reported': dates['enacted_first_reported'],
        'Date_Last_Amended': dates['last_amended'],
        'Expiry_End_Date': dates['expiry_end'],
        'Date_Last_Extension': dates['last_extension'],
        'Other_Date': dates['other_date'],
    }
    for cat in BPM6_CATS:
        rec[f'{cat}_effect']    = effects[cat]
        rec[f'{cat}_direction'] = direction if effects[cat] == 1 else 0
    for cat in BPM6_CATS:
        rec[f'{cat}_reasoning'] = reasoning[cat]
    results.append(rec)

print(f'Done: {len(results):,} regulations')

      0 / 6,992
  1,000 / 6,992
  2,000 / 6,992
  3,000 / 6,992
  4,000 / 6,992
  5,000 / 6,992
  6,000 / 6,992
Done: 6,992 regulations


## 10 · Summary Statistics

In [12]:
print(f"{'Category':<50s} {'Affected':>8s}  {'Pos':>5s}  {'Neg':>5s}")
print('-'*72)
for cat in BPM6_CATS:
    n   = sum(1 for r in results if r[f'{cat}_effect']==1)
    pos = sum(1 for r in results if r[f'{cat}_direction']==1)
    neg = sum(1 for r in results if r[f'{cat}_direction']==-1)
    print(f'{cat:<50s} {n:>8,}  {pos:>5,}  {neg:>5,}')

Category                                           Affected    Pos    Neg
------------------------------------------------------------------------
General_merchandise                                   2,168    373  1,795
Goods_under_merchanting                                   8      0      8
Nonmonetary_gold                                          4      0      4
Manufacturing_services                                    5      0      5
Maintenance_repair_services                              30      1     29
Transport                                               240     42    198
Travel                                                   29      1     28
Construction                                             29      2     27
Insurance_pension_services                               49      6     43
Financial_services                                      481     47    434
IP_charges                                            1,449    581    868
Telecom_computer_info_services         

## 11 · Disaggregate Telecom into BPM6 9.1 / 9.2 / 9.3

The aggregate `Telecom_computer_info_services` flag is now split into three
BPM6 subcomponents per §10.141–10.146:

* **9.1 Telecommunications** — transmission of sound, data, images (telephone,
  mobile, internet access, broadcast/satellite transmission, ISP backbone).
  Excludes the content transmitted.
* **9.2 Computer services** — software (custom + mass-market), hosting, data
  processing, IT consultancy, hardware/software install & maintenance, web
  hosting, web page design, application/cloud services. Excludes pure content.
* **9.3 Information services** — news agency services, databases, search
  portals, online content provision, libraries/archives, direct subscriptions
  to electronically-delivered newspapers/periodicals.

Disaggregation is rule-based on `(Pillar, Subpillar, Coverage)`. For each row
where the aggregate effect is `1`, a deterministic mapping table assigns one,
two, or all three subcomponent flags. Coverage keywords act as overrides that
can *add* (never remove) subcomponents.

In [13]:
# Subpillar -> default (9.1, 9.2, 9.3) flags for telecom-affected rows
_SUB_TO_SUBCOMP = {}
def _add(sp, a, b, c):
    _SUB_TO_SUBCOMP[sp.strip().lower()] = (a, b, c)

# Pillar: Telecom infrastructure & competition  -> 9.1 only
for sp in [
    "Presence of an independent telecom authority",
    "Passive infrastructure sharing obligation",
    "Functional/accounting separation for operators with significant market power",
    "Signature of the WTO Telecom Reference Paper",
    "Presence of shares owned by the government in telecom companies",
    "Licensing restrictions to operate in the telecom market",
    "Maximum foreign equity share for investment in the telecommunication sector",
    "Presence of Internet shutdowns",
]:
    _add(sp, 1, 0, 0)

# Pillar: Cross-border data policies  -> 9.2 + 9.3
for sp in [
    "Ban to transfer and local processing requirement",
    "Conditional flow regime",
    "Local storage requirement",
    "Participation in trade agreements committing to open cross-border data flows",
]:
    _add(sp, 0, 1, 1)

# Pillar: Domestic data policies
_add("Framework for data protection", 0, 1, 1)
_add("Minimum period for data retention", 1, 1, 0)
_add("Requirement to allow the government to access personal data collected", 1, 1, 0)
_add("Requirement to perform a Data Protection Impact Assessment (DPIA) or have a data protection officer (DPO)", 0, 1, 0)
_add("User identity requirement", 1, 1, 1)

# Pillar: Intermediary liability  -> 9.2 + 9.3
for sp in [
    "Safe harbour for intermediaries for copyright infringement",
    "Safe harbour for intermediaries for any activity other than copyright infringement",
    "Monitoring requirement",
    "Enforcement of copyright online",
]:
    _add(sp, 0, 1, 1)

# Pillar: Content access
_add("Blocking or filtering of commercial web content", 0, 0, 1)
_add("Restrictions on online advertising", 0, 0, 1)
_add("Restrictions on domain names", 0, 1, 1)
_add("Restrictions on encryption standards", 1, 1, 0)

# Pillar: Online sales & transactions
_add("Framework for consumer protection applicable to online commerce", 0, 1, 1)
_add("Ratification of the UN Convention on the Use of Electronic Communications in International Contracts", 0, 1, 0)
_add("UNCITRAL Model Law on Electronic Commerce", 0, 1, 0)
_add("UNCITRAL Model Law on Electronic Signatures", 0, 1, 0)
_add("Restrictions on online payments", 0, 1, 0)
_add("Licensing scheme for e-commerce providers", 0, 1, 1)
_add("Limits on e-commerce purchases", 0, 1, 1)
_add("Maximum foreign equity share for investment in the e-commerce sector", 0, 1, 1)

# Pillar: FDI -> coverage-driven
for sp in [
    "Maximum foreign equity share",
    "Screening of investment and acquisitions",
    "Nationality/residency requirement for directors or managers",
    "Requirement to engage in joint ventures to invest or operate",
]:
    _SUB_TO_SUBCOMP[sp.strip().lower()] = "FDI"

# Pillar: Intellectual Property Rights
_add("Practical or legal restrictions related to the application process for patents", 0, 1, 0)
_add("Practical or legal restrictions related to the enforcement of patents", 0, 1, 0)
_add("Copyright law with clear exceptions", 0, 1, 1)
_add("Effective protection covering trade secrets", 0, 1, 0)
_add("Mandatory disclosure of business trade secrets such as algorithms or source code", 0, 1, 0)
_add("Participation in the Patent Cooperation Treaty (PCT)", 0, 1, 0)

# Pillar: Public procurement
for sp in [
    "Other limitations on foreign participation in public procurement",
    "Exclusion from public procurement",
    "Local content requirements (LCRs) on ICT goods for the commercial market",
    "Surrender of patents, source code or trade secrets to win public tenders/Restrictions on technology standards for public tenders",
]:
    _add(sp, 0, 1, 0)
_add("Signatory of the WTO Agreement on Government Procurement (GPA) with coverage of the most relevant services sectors (CPC 752, 754, 84)", 1, 1, 0)

# Pillar: Quantitative trade restrictions for ICT goods
_add("Other import restrictions, including non-transparent/discriminatory import procedures", 0, 1, 0)
_add("Self-certification for product safety", 0, 1, 0)
_add("Threshold for ‘De Minimis’ rule", 0, 1, 0)
_add("Export restrictions on ICT goods or online services", 0, 1, 0)
_add("Import ban applied on ICT goods or online services", 0, 1, 0)
_add("Licensing schemes for digital services and applications", 0, 1, 1)
_add("Local presence requirements for digital services providers", 0, 1, 1)
_add("Commercial presence requirement for digital services providers", 0, 1, 1)
_add("Infrastructure requirement", 1, 1, 0)

# Pillar: Technical standards
_add("Open and transparent standard-setting process", 0, 1, 0)
_add("Product screening and additional testing requirements", 0, 1, 0)

# Pillar: Tariffs & trade defence
_add("Antidumping, countervailing duties, and safeguard measures on ICT goods", 0, 1, 0)

# Coverage-keyword overrides (can add components, never remove)
_COV_91 = re.compile(
    r"\btelecom|telecommunication|mobile (?:network|operator|sector)|"
    r"internet (?:access|service provider|backbone)|broadband|isp\b|"
    r"satellite (?:transmission|communication)|broadcast(?:ing)?\s+(?:transmission|sector|services?)|"
    r"cellular|wireless (?:network|infrastructure)|fixed[- ]line", re.I)
_COV_93 = re.compile(
    r"\bnews (?:agency|agencies|website|websites|media)|press|online press|"
    r"media (?:platform|content|outlet|intermediar)|broadcasting content|"
    r"publisher|content[- ]sharing|search (?:engine|portal)|"
    r"database (?:provider|service)|information (?:provision|services)|"
    r"online content|digital content (?:producer|provider)|"
    r"online advertising|advertising platform", re.I)
_COV_92 = re.compile(
    r"\bsoftware|cloud|hosting|data center|data centre|data processing|"
    r"it services?|computer services?|web (?:hosting|design)|application|"
    r"saas|paas|iaas|fintech|payment (?:service|processor|system|platform)|"
    r"e-?commerce|digital (?:platform|service|application)|"
    r"online (?:platform|application|service)|encryption (?:product|software|standard)",
    re.I)


def disaggregate_telecom(rec):
    """Compute (e91, d91, e92, d92, e93, d93) for one record."""
    if rec["Telecom_computer_info_services_effect"] != 1:
        return 0, 0, 0, 0, 0, 0

    sp = str(rec["Subpillar"]).strip().lower()
    cov = str(rec["Coverage"]) if pd.notna(rec["Coverage"]) else ""
    direction = rec["Telecom_computer_info_services_direction"]

    rule = _SUB_TO_SUBCOMP.get(sp)

    if rule == "FDI":
        if cov.strip().lower() == "horizontal":
            e91, e92, e93 = 1, 1, 1
        else:
            e91 = 1 if _COV_91.search(cov) else 0
            e92 = 1 if _COV_92.search(cov) else 0
            e93 = 1 if _COV_93.search(cov) else 0
            if not (e91 or e92 or e93):
                e91, e92, e93 = 1, 1, 1
    elif rule is None:
        e91 = 1 if _COV_91.search(cov) else 0
        e92 = 1 if _COV_92.search(cov) else 0
        e93 = 1 if _COV_93.search(cov) else 0
        if not (e91 or e92 or e93):
            e92 = 1
    else:
        e91, e92, e93 = rule
        if _COV_91.search(cov): e91 = 1
        if _COV_92.search(cov): e92 = 1
        if _COV_93.search(cov): e93 = 1

    d91 = direction if e91 else 0
    d92 = direction if e92 else 0
    d93 = direction if e93 else 0
    return e91, d91, e92, d92, e93, d93


_BPM6_91 = ("BPM6 9.1 = transmission of sound/data/images: telephone, mobile, "
            "internet access, broadcast/satellite transmission, ISP backbone "
            "(excludes content).")
_BPM6_92 = ("BPM6 9.2 = computer services: software, hosting, data processing, "
            "IT consultancy, web/cloud services.")
_BPM6_93 = ("BPM6 9.3 = information services: news agencies, databases, search "
            "portals, online content provision.")


def disaggregate_telecom_reasoning(rec, e91, e92, e93):
    pillar = str(rec["Pillar"]) if pd.notna(rec["Pillar"]) else ""
    sp = str(rec["Subpillar"]) if pd.notna(rec["Subpillar"]) else ""
    cov = str(rec["Coverage"]) if pd.notna(rec["Coverage"]) else ""
    orig_eff = rec["Telecom_computer_info_services_effect"]
    orig_dir = rec["Telecom_computer_info_services_direction"]
    orig_reason = rec.get("Telecom_computer_info_services_reasoning", "")

    if orig_eff != 1:
        return (orig_reason + " | Subcomponent disaggregation: original aggregate "
                "effect was 0; all of 9.1/9.2/9.3 = 0 (no further assessment).")

    dir_label = "positive (enabling)" if orig_dir == 1 else "negative (restrictive)"
    parts = [
        f"Disaggregated from original aggregate (effect=1, {dir_label}). "
        f"Pillar: '{pillar}'; Subpillar: '{sp}'; "
        f"Coverage: '{cov[:100]}{'...' if len(cov) > 100 else ''}'."
    ]

    sp_l = sp.lower()
    cov_l = cov.lower()

    # 9.1
    if e91:
        if pillar.startswith("Telecom infrastructure"):
            parts.append(f"9.1 AFFECTED: subpillar regulates telecom operators / "
                         f"transmission infrastructure directly. {_BPM6_91}")
        elif any(k in cov_l for k in ["telecom", "internet access", "broadband",
                                       "isp", "mobile", "satellite transmission"]):
            parts.append(f"9.1 AFFECTED: Coverage explicitly references telecom "
                         f"transmission services. {_BPM6_91}")
        elif "data retention" in sp_l or "government to access" in sp_l:
            parts.append(f"9.1 AFFECTED: imposes obligations on telecom operators "
                         f"(lawful intercept / metadata retention). {_BPM6_91}")
        elif "user identity" in sp_l:
            parts.append(f"9.1 AFFECTED: SIM registration / KYC affects telecom "
                         f"subscribers. {_BPM6_91}")
        elif "encryption" in sp_l:
            parts.append(f"9.1 AFFECTED: encryption rules affect telecom "
                         f"transmission. {_BPM6_91}")
        elif "infrastructure requirement" in sp_l:
            parts.append(f"9.1 AFFECTED: infrastructure requirements impact "
                         f"telecom networks. {_BPM6_91}")
        else:
            parts.append(f"9.1 AFFECTED based on Coverage/Subpillar context. {_BPM6_91}")
    else:
        parts.append("9.1 NOT affected: regulation does not concern transmission "
                     "services per BPM6 9.1.")

    # 9.2
    if e92:
        if pillar.startswith("Cross-border data") or pillar.startswith("Domestic data") or "data" in sp_l:
            parts.append(f"9.2 AFFECTED: data policies affect cloud/hosting/"
                         f"data-processing services. {_BPM6_92}")
        elif pillar.startswith("Intermediary"):
            parts.append(f"9.2 AFFECTED: intermediary liability affects hosting "
                         f"providers. {_BPM6_92}")
        elif pillar.startswith("Online sales"):
            parts.append(f"9.2 AFFECTED: e-commerce framework affects digital "
                         f"platforms / payment systems. {_BPM6_92}")
        elif (pillar.startswith("Public procurement") or pillar.startswith("Quantitative")
              or pillar.startswith("Tariffs") or pillar.startswith("Technical standards")):
            parts.append(f"9.2 AFFECTED: regulation on ICT goods/services materially "
                         f"affects computer services delivery. {_BPM6_92}")
        elif pillar.startswith("Intellectual Property"):
            parts.append(f"9.2 AFFECTED: IPR rules apply to software / source code / "
                         f"algorithms. {_BPM6_92}")
        elif pillar.startswith("Foreign Direct Investment"):
            parts.append(f"9.2 AFFECTED: FDI restrictions touch the computer/"
                         f"digital services sector. {_BPM6_92}")
        elif "domain" in sp_l:
            parts.append(f"9.2 AFFECTED: domain registration is a computer "
                         f"service. {_BPM6_92}")
        elif "encryption" in sp_l:
            parts.append(f"9.2 AFFECTED: encryption standards affect software "
                         f"products. {_BPM6_92}")
        else:
            parts.append(f"9.2 AFFECTED based on Coverage/Subpillar context. {_BPM6_92}")
    else:
        parts.append("9.2 NOT affected: regulation does not concern computer "
                     "services per BPM6 9.2.")

    # 9.3
    if e93:
        if pillar.startswith("Content access") and ("blocking" in sp_l or "advertising" in sp_l):
            parts.append(f"9.3 AFFECTED: directly governs online content / "
                         f"information distribution. {_BPM6_93}")
        elif pillar.startswith("Intermediary"):
            parts.append(f"9.3 AFFECTED: intermediary rules affect online content/"
                         f"information platforms. {_BPM6_93}")
        elif pillar.startswith("Cross-border data") or pillar.startswith("Domestic data"):
            parts.append(f"9.3 AFFECTED: data rules affect database/information "
                         f"service providers. {_BPM6_93}")
        elif pillar.startswith("Online sales"):
            parts.append(f"9.3 AFFECTED: e-commerce platforms host product "
                         f"information / catalogues. {_BPM6_93}")
        elif "domain" in sp_l:
            parts.append(f"9.3 AFFECTED: domain restrictions affect information "
                         f"portals' addressing. {_BPM6_93}")
        elif pillar.startswith("Foreign Direct Investment"):
            parts.append(f"9.3 AFFECTED: FDI restrictions touch online content / "
                         f"news / media sector. {_BPM6_93}")
        elif "user identity" in sp_l:
            parts.append(f"9.3 AFFECTED: user identity rules apply to content / "
                         f"information platforms. {_BPM6_93}")
        elif "copyright" in sp_l or "intellectual property" in pillar.lower():
            parts.append(f"9.3 AFFECTED: copyright rules affect digital content "
                         f"distribution. {_BPM6_93}")
        else:
            parts.append(f"9.3 AFFECTED based on Coverage/Subpillar context. {_BPM6_93}")
    else:
        parts.append("9.3 NOT affected: regulation does not concern news/database/"
                     "information provision per BPM6 9.3.")

    parts.append(f"Direction inherited from original aggregate: {orig_dir} ({dir_label}); "
                 f"no row-level evidence justifies a different direction across subcomponents.")
    return " ".join(parts)


for rec in results:
    e91, d91, e92, d92, e93, d93 = disaggregate_telecom(rec)
    rec["Telecom_9_1_effect"]    = e91
    rec["Telecom_9_1_direction"] = d91
    rec["Telecom_9_2_effect"]    = e92
    rec["Telecom_9_2_direction"] = d92
    rec["Telecom_9_3_effect"]    = e93
    rec["Telecom_9_3_direction"] = d93
    rec["Telecom_9_1_9_2_9_3_reasoning"] = disaggregate_telecom_reasoning(rec, e91, e92, e93)

n_91 = sum(1 for r in results if r["Telecom_9_1_effect"] == 1)
n_92 = sum(1 for r in results if r["Telecom_9_2_effect"] == 1)
n_93 = sum(1 for r in results if r["Telecom_9_3_effect"] == 1)
print(f"Telecom subcomponent counts: 9.1={n_91:,}  9.2={n_92:,}  9.3={n_93:,}")


Telecom subcomponent counts: 9.1=2,088  9.2=4,125  9.3=2,242


## 12 · Partner-Specific Applicability Scan

Some regulations apply only to specific partner countries or country groups
(reciprocity clauses, sanctions, regional-bloc preferences, bilateral
carve-outs, GPA-only procurement eligibility, etc.). A strict regex scan over
`Coverage` and `Description` flags these rows. Pure descriptive entries of the
form *"Country X is/is not a signatory of Y"* are filtered out as they describe
the country's own status, not partner-discriminatory treatment.

In [14]:
_PARTNER_DISC = re.compile(
    r"(\breciproc(?:al|ity)\b"
    r"|\bsanction(?:ed|s|ing)\b|\bembargo\b"
    r"|\bcountries with which\b"
    r"|\bonly applies to (?:nationals|citizens|companies|firms|investors|persons) (?:of|from)\b"
    r"|\bdoes not apply to (?:nationals|citizens|companies|firms|investors|persons|residents) (?:of|from)\b"
    r"|\bexcept (?:for )?(?:nationals|citizens|companies|firms|investors|residents|persons) (?:of|from) [A-Z]"
    r"|\bpreferential (?:treatment|access|tariff|rate)\b"
    r"|\bFTA partners?\b|\bfree trade partners?\b"
    r"|\b(?:United States|U\.S\.|China|Russia|Iran|North Korea|Cuba|Syria|Venezuela|Belarus)\s+"
    r"(?:companies|nationals|investors|firms|entities|origin|persons|individuals|goods)"
    r"|\bnot apply to .{0,40}(?:EU|EEA|EFTA|ASEAN|GCC|Mercosur|member states?)"
    r"|\bnationality.{0,30}(?:exempt|priority|preference|preferential|requirement)"
    r"|\b(?:EU|EEA|EFTA|ASEAN|GCC|Mercosur|EAEU|GPA|WTO Government Procurement Agreement)\s+"
    r"(?:countries|members?|nationals|companies|investors|firms)"
    r"|\bcountry of origin\b"
    r"|\bnot less favourable\b"
    r"|\bbilateral agreement(?:s)?\b"
    r"|\bbilateral treat(?:y|ies)\b"
    r"|\bGovernment Procurement Agreement\b"
    r")",
    re.IGNORECASE,
)
_PARTNER_DESCRIPTIVE = re.compile(
    r"^\s*\w+\s+(?:is|has|are|have)\s+(?:not\s+)?(?:a\s+)?"
    r"(?:signator|signed|ratif|enact|adopt|implement|member|"
    r"part(?:y|ies)?\s+to|join(?:ed)?)",
    re.IGNORECASE,
)


def detect_partner_specific(rec):
    desc = str(rec.get("Description", "") or "")
    cov  = str(rec.get("Coverage", "")    or "")
    text = cov + " || " + desc
    m = _PARTNER_DISC.search(text)
    is_descriptive = bool(_PARTNER_DESCRIPTIVE.match(desc.strip())) and len(desc) < 250
    if m and not is_descriptive:
        start = max(0, m.start() - 60)
        end   = min(len(text), m.end() + 80)
        return 1, text[start:end].strip()
    return 0, ""


for rec in results:
    flag, evidence = detect_partner_specific(rec)
    rec["Partner_Specific"] = flag
    if flag:
        ev = evidence.replace("\n", " ").strip()
        rec["Partner_Specific_reasoning"] = (
            f"Partner_Specific=1. Evidence from Description/Coverage indicating "
            f"limited applicability to specific partner countries or country "
            f"groups: \"...{ev}...\""
        )
    else:
        rec["Partner_Specific_reasoning"] = (
            "Partner_Specific=0. No language indicating partner-specific "
            "applicability; regulation applies universally."
        )

n_partner = sum(1 for r in results if r["Partner_Specific"] == 1)
print(f"Partner_Specific = 1 count: {n_partner:,} / {len(results):,}")


Partner_Specific = 1 count: 231 / 6,992


## 13 · EU Regulation Disaggregation Flag

The dataset contains 29 rows where `Country = "EUROPEAN UNION"`. For
downstream analyses that operate at the member-state level, each EU row is
a candidate for *disaggregation* — cloning the row to each of the 27 member
states the regulation actually binds. Naïve disaggregation creates duplicates
because:

1. **GDPR** (Reg. (EU) 2016/679) appears at every member state's row under
   *Framework for data protection*. The EU rows under *Local presence
   requirements*, *Conditional flow regime*, and *DPIA / DPO requirement*
   would duplicate that content.
2. **Trade Secrets Directive** (Directive (EU) 2016/943) is transposed at
   every member state's row under *Effective protection covering trade secrets*.
3. **Data Act** (Reg. (EU) 2023/2854) overlaps with GDPR transfer rules under
   *Conditional flow regime*.
4. **Tariff statistics rows** (effective tariff rate, zero-tariff coverage)
   contain *country-specific* values — disaggregating the EU figure would
   overwrite each member state's WITS-derived value.
5. **Three further EU rows** partially collide:
   - DSA local-presence and DSA mandatory-disclosure rows collide with
     Germany's existing DSA row.
   - 2008 New Legislative Framework collides with Poland's existing row.

The 22 remaining EU rows have no member-state collision and can be
disaggregated cleanly. This section adds two outputs:

* A `Disaggregate` flag column on the Regulation Mapping sheet:
  `1` for the 22 EU rows that should be disaggregated, `0` for everything
  else (including the 7 EU rows that should not be disaggregated).
* A separate `EU_Disaggregation_Reasoning` sheet with one row per EU
  regulation, recording the decision, the reasoning, the member-state
  collisions, and the explicit list of countries to disaggregate to.

Identification is done by content (Subpillar + regulation IDs detected in
Act/Practice, Coverage, and Description), so the logic is robust to row
reordering.

In [ ]:
# 27 EU member states (uppercase to match dataset's Country column)
EU27 = [
    "AUSTRIA", "BELGIUM", "BULGARIA", "CROATIA", "CYPRUS", "CZECH REPUBLIC",
    "DENMARK", "ESTONIA", "FINLAND", "FRANCE", "GERMANY", "GREECE", "HUNGARY",
    "IRELAND", "ITALY", "LATVIA", "LITHUANIA", "LUXEMBOURG", "MALTA",
    "NETHERLANDS", "POLAND", "PORTUGAL", "ROMANIA", "SLOVAKIA", "SLOVENIA",
    "SPAIN", "SWEDEN",
]
ALL27 = ", ".join(EU27)
ALL_EXCEPT_DE = ", ".join(c for c in EU27 if c != "GERMANY")
ALL_EXCEPT_PL = ", ".join(c for c in EU27 if c != "POLAND")
NONE_TEXT = "None — do not disaggregate to any member state."


# Decision dictionary keyed by short identifier. Each value records the
# disaggregation flag, decision category, reasoning, member-state collisions,
# and the explicit list of countries to disaggregate to (column L).
EU_DECISIONS = {
    # ===== DO NOT DISAGGREGATE — full collisions =====
    "GDPR_LOCAL_PRESENCE": dict(
        disaggregate=0,
        category="Full collision — do NOT disaggregate",
        reasoning=("GDPR (Regulation (EU) 2016/679) is directly applicable in every EU "
                   "member state, but the DTI dataset already records each of the 27 "
                   "members' national implementation explicitly under 'Framework for "
                   "data protection' (e.g., Germany's Federal Data Protection Act, "
                   "France's Act No. 78-17, Italy's Personal Data Protection Code "
                   "D.lgs. 196/2003, etc.). The member-state rows are richer than the "
                   "EU row because they cite both GDPR and the national implementation act."),
        collisions=("All 27 member states. Disaggregating would duplicate substantive "
                    "content already captured at member level."),
        action=NONE_TEXT,
    ),
    "GDPR_CONDITIONAL": dict(
        disaggregate=0,
        category="Full collision — do NOT disaggregate",
        reasoning=("GDPR is already captured at member-state level for the "
                   "'Conditional flow regime' subpillar. Disaggregating the EU row "
                   "would create a duplicate of substantive content already present "
                   "in the dataset under each of the 27 member states."),
        collisions=("All 27 member states have GDPR-related rows under 'Framework for "
                    "data protection' or 'Conditional flow regime' (or both)."),
        action=NONE_TEXT,
    ),
    "DATA_ACT": dict(
        disaggregate=0,
        category="Full collision — do NOT disaggregate",
        reasoning=("This row references Regulation (EU) 2023/2854 (Data Act, applicable "
                   "from 12 September 2025) together with GDPR. Although the Data Act "
                   "adds new content beyond GDPR, its conditional-flow-regime aspect "
                   "overlaps materially with GDPR transfer rules already captured at "
                   "every member state. Disaggregating creates duplicates with existing "
                   "GDPR-based member rows."),
        collisions="All 27 member states (via GDPR coverage on the same subpillar).",
        action=NONE_TEXT,
    ),
    "GDPR_DPIA": dict(
        disaggregate=0,
        category="Full collision — do NOT disaggregate",
        reasoning=("GDPR DPIA / DPO requirements (Arts. 35 and 37-39) are captured at "
                   "member-state level. The DTI records national implementing "
                   "legislation for each of the 27 members. Disaggregating duplicates "
                   "content."),
        collisions="All 27 member states.",
        action=NONE_TEXT,
    ),
    "TRADE_SECRETS_DIR": dict(
        disaggregate=0,
        category="Full collision — do NOT disaggregate",
        reasoning=("Directive (EU) 2016/943 (Trade Secrets Directive) has been "
                   "transposed into national law by all 27 member states, and the DTI "
                   "dataset captures every transposition explicitly under 'Effective "
                   "protection covering trade secrets' (e.g., Germany's Law transposing "
                   "Directive (EU) 2016/943, Spain's Law 1/2019, Italy's Decreto "
                   "Legislativo 63/2018, etc.). The member-state rows are richer "
                   "because they cite both the Directive and the national implementing law."),
        collisions="All 27 member states.",
        action=NONE_TEXT,
    ),
    "TARIFF_RATE": dict(
        disaggregate=0,
        category="Country-specific data — do NOT disaggregate",
        reasoning=("Although the EU operates a Common External Tariff under TFEU "
                   "Art. 31, the DTI dataset records a country-specific applied "
                   "weighted-average tariff value for each of the 27 member states "
                   "(e.g., Germany 0.83, France 0.89, Latvia 0.35, Croatia 1.15, all "
                   "distinct from the EU's 0.80). The country-level numbers come from "
                   "WITS computations that reflect each member's actual import "
                   "composition. Disaggregating the EU figure to each member would "
                   "overwrite the existing country-specific values and corrupt the data."),
        collisions=("All 27 member states have their own country-specific values for "
                    "this metric."),
        action=NONE_TEXT,
    ),
    "TARIFF_ZERO": dict(
        disaggregate=0,
        category="Country-specific data — do NOT disaggregate",
        reasoning=("Same logic as Row 25. Each member state has a country-specific "
                   "zero-tariff coverage rate from WITS, distinct from the EU aggregate."),
        collisions="All 27 member states have their own country-specific values.",
        action=NONE_TEXT,
    ),

    # ===== PARTIAL COLLISIONS =====
    "DSA_LOCAL_PRESENCE": dict(
        disaggregate=1,
        category="Partial collision — disaggregate with one exclusion",
        reasoning=("Regulation (EU) 2022/2065 (Digital Services Act) is a directly "
                   "applicable EU Regulation that has bound all 27 member states from "
                   "17 February 2024 without national transposition under TFEU Art. 288. "
                   "Substantively the regulation applies in each member state, so "
                   "disaggregation reflects the legal reality."),
        collisions=("Germany — already has 1 row in the DTI dataset that explicitly "
                    "references the DSA. Disaggregating the EU row to all 27 members "
                    "would create a duplicate Germany entry."),
        action=ALL_EXCEPT_DE,
    ),
    "DSA_DISCLOSURE": dict(
        disaggregate=1,
        category="Partial collision — disaggregate with one exclusion",
        reasoning=("Same legal logic as Row 1: the DSA is directly applicable in all "
                   "27 member states from 17 February 2024. The mandatory-disclosure "
                   "aspect (Reg. (EU) 2022/2065 transparency obligations for "
                   "VLOPs/VLOSEs) applies uniformly."),
        collisions="Germany — same DSA collision noted for Row 1.",
        action=ALL_EXCEPT_DE,
    ),
    "NLF_STANDARD_SETTING": dict(
        disaggregate=1,
        category="Partial collision — disaggregate with one exclusion",
        reasoning=("The 2008 New Legislative Framework (NLF) — comprising Regulation "
                   "(EC) 765/2008 and Decision 768/2008/EC — sets the EU's horizontal "
                   "framework for accreditation, market surveillance, and conformity "
                   "assessment. It applies across all 27 member states."),
        collisions=("Poland — has 1 row in the DTI dataset under the same subpillar "
                    "('Open and transparent standard-setting process'). Disaggregating "
                    "the EU row to all 27 would create a duplicate Poland entry."),
        action=ALL_EXCEPT_PL,
    ),

    # ===== CLEAN DISAGGREGATION =====
    "DE_MINIMIS": dict(
        disaggregate=1,
        category="Clean disaggregation",
        reasoning=("Council Directive (EU) 2017/2455 amended the EU VAT Directive and "
                   "removed the low-value consignment relief at the EU's external "
                   "border, replacing it with the IOSS scheme effective 1 July 2021. "
                   "As a customs-union-wide measure adopted under TFEU Arts. 113 and "
                   "207, it applies uniformly to all 27 member states' external borders."),
        collisions="None — no member-state row in the DTI dataset references this regulation.",
        action=ALL27,
    ),
    "NIS_LOCAL_PRESENCE": dict(
        disaggregate=1,
        category="Clean disaggregation (no captured collisions)",
        reasoning=("Directive (EU) 2016/1148 (NIS Directive) required transposition by "
                   "all 27 member states by 9 May 2018 and has since been replaced by "
                   "Directive (EU) 2022/2555 (NIS2 Directive), with a transposition "
                   "deadline of 17 October 2024. The directive substantively applies in "
                   "every member state via national implementing legislation."),
        collisions=("None measured — although every member state has in fact transposed "
                    "the NIS / NIS2 Directive, the DTI dataset does not capture these "
                    "national implementations under the 'Local presence requirements "
                    "for digital services providers' subpillar."),
        action=ALL27,
    ),
    "RUSSIA_BLOCKING": dict(
        disaggregate=1,
        category="Clean disaggregation (CFSP / customs union)",
        reasoning=("Council Regulation (EU) 2022/350 amended Regulation (EU) 833/2014 "
                   "to suspend broadcasting activities of RT and Sputnik in the EU. As "
                   "a Council Regulation under TFEU Art. 215 (CFSP-implementing "
                   "measures), it is directly applicable in all 27 member states "
                   "without national transposition."),
        collisions="None — no member-state row captures this measure.",
        action=ALL27,
    ),
    "DUAL_USE_EXPORT": dict(
        disaggregate=1,
        category="Clean disaggregation (customs union)",
        reasoning=("Regulation (EU) 2021/821 sets the EU's dual-use export control "
                   "regime. The Common Commercial Policy under TFEU Art. 207 makes "
                   "export controls an exclusive Union competence; the regulation "
                   "applies uniformly at all 27 member states' external borders. "
                   "Although member states retain national licensing authorities and "
                   "ancillary national legislation, the DTI dataset does not capture "
                   "these national implementations under the export-restrictions "
                   "subpillar."),
        collisions="None — no member-state row references this regulation.",
        action=ALL27,
    ),
    "RUSSIA_EXPORT": dict(
        disaggregate=1,
        category="Clean disaggregation (CFSP / customs union)",
        reasoning=("Council Regulation (EU) 833/2014, as amended throughout 2022-2024, "
                   "imposes EU-wide export restrictions on Russia. Adopted under TFEU "
                   "Art. 215, it is directly applicable in all 27 member states."),
        collisions="None — no member-state row captures this measure.",
        action=ALL27,
    ),
    "BELARUS_EXPORT": dict(
        disaggregate=1,
        category="Clean disaggregation (CFSP / customs union)",
        reasoning=("Council Regulation (EU) 2023/1529 imposes EU-wide export "
                   "restrictions on Belarus. Like Reg. 833/2014, it is a directly "
                   "applicable Council Regulation adopted under TFEU Art. 215."),
        collisions="None — no member-state row captures this measure.",
        action=ALL27,
    ),
    "SELF_CERT": dict(
        disaggregate=1,
        category="Clean disaggregation (single market)",
        reasoning=("Regulation (EU) 2019/515 on the mutual recognition of goods is "
                   "directly applicable across all 27 member states and is foundational "
                   "to free movement of goods under TFEU Arts. 34-36. The "
                   "Electromagnetic Compatibility Directive 2014/30/EU referenced in "
                   "the row is also fully harmonised across the EU."),
        collisions="None — no member-state row in this subpillar.",
        action=ALL27,
    ),
    "WTO_TELECOM": dict(
        disaggregate=1,
        category="Clean disaggregation (no member rows)",
        reasoning=("Each EU member state is independently a WTO member and made its "
                   "own GATS schedule of commitments, including its own appendment of "
                   "the Telecom Reference Paper. The DTI dataset captures this only at "
                   "EU level. Disaggregation to all 27 reflects each member's "
                   "individual GATS commitment; no member-state row exists to collide with."),
        collisions="None — no member-state row captures this commitment.",
        action=ALL27,
    ),
    "PNR": dict(
        disaggregate=1,
        category="Clean disaggregation (no captured collisions)",
        reasoning=("Directive (EU) 2016/681 (PNR Directive) required national "
                   "transposition by all 27 member states by 25 May 2018. While the "
                   "directive substantively applies in every member state via national "
                   "implementing legislation, the DTI dataset does not capture these "
                   "national transpositions in this subpillar."),
        collisions="None measured.",
        action=ALL27,
    ),
    "EU_UK_TCA": dict(
        disaggregate=1,
        category="Clean disaggregation (Union-level FTA)",
        reasoning=("The EU-UK Trade and Cooperation Agreement is a Union-level mixed "
                   "agreement with binding cross-border data flow commitments (Title "
                   "III, Chapter 1). Under TFEU Art. 207 and ECJ Opinion 2/15, common "
                   "commercial policy is an exclusive Union competence; all 27 member "
                   "states are bound by the TCA's data-flow provisions."),
        collisions="None — no member-state row captures this commitment.",
        action=ALL27,
    ),
    "ANTIDUMPING_2015": dict(
        disaggregate=1,
        category="Clean disaggregation (customs union)",
        reasoning=("Commission Implementing Regulation (EU) 2015/1953 (and its 2022 "
                   "extension Reg. 2022/58) imposes definitive antidumping duties under "
                   "the EU's Common Commercial Policy. Antidumping is an exclusive "
                   "Union competence under TFEU Art. 207; the duties apply at all 27 "
                   "member states' external borders."),
        collisions="None — no member-state row captures EU antidumping measures.",
        action=ALL27,
    ),
    "ANTIDUMPING_SILICON": dict(
        disaggregate=1,
        category="Clean disaggregation (customs union)",
        reasoning=("EU antidumping measure on silicon metal. Same customs-union logic "
                   "as Row 18."),
        collisions="None.",
        action=ALL27,
    ),
    "ANTIDUMPING_2024": dict(
        disaggregate=1,
        category="Clean disaggregation (customs union)",
        reasoning=("Commission Implementing Regulation (EU) 2024/3014 imposing "
                   "antidumping duties on optical fibre cables. Same customs-union "
                   "logic as Row 18."),
        collisions="None.",
        action=ALL27,
    ),
    "ANTIDUMPING_2021_2011": dict(
        disaggregate=1,
        category="Clean disaggregation (customs union)",
        reasoning=("Commission Implementing Regulations (EU) 2021/2011 and 2022/72 "
                   "imposing antidumping duties on optical fibre cables originating in "
                   "China. Same customs-union logic as Row 18."),
        collisions="None.",
        action=ALL27,
    ),
    "ANTIDUMPING_2022_72": dict(
        disaggregate=1,
        category="Clean disaggregation (customs union)",
        reasoning=("Commission Implementing Regulation (EU) 2022/72. Same customs-union "
                   "logic as Row 18."),
        collisions="None.",
        action=ALL27,
    ),
    "IPI": dict(
        disaggregate=1,
        category="Clean disaggregation",
        reasoning=("Regulation (EU) 2022/1031 (International Procurement Instrument) "
                   "is a directly applicable EU Regulation governing access of "
                   "third-country economic operators to EU public procurement above "
                   "specified thresholds. It applies in all 27 member states from "
                   "29 August 2022."),
        collisions="None — no member-state row captures this measure.",
        action=ALL27,
    ),
    "WTO_GPA": dict(
        disaggregate=1,
        category="Clean disaggregation (EU-level WTO commitment)",
        reasoning=("The EU is party to the WTO Agreement on Government Procurement on "
                   "behalf of all 27 member states with a single schedule of "
                   "commitments. The lack of CPC 754 (telecom-related services) "
                   "coverage in the EU's GPA schedule applies to each member state's "
                   "procurement market."),
        collisions="None — no member-state row captures this commitment.",
        action=ALL27,
    ),
    "P2B": dict(
        disaggregate=1,
        category="Clean disaggregation",
        reasoning=("Regulation (EU) 2019/1150 (Platform-to-Business Regulation) is a "
                   "directly applicable EU Regulation in force since 12 July 2020. It "
                   "applies to all 27 member states without national transposition."),
        collisions="None — no member-state row captures this regulation.",
        action=ALL27,
    ),
    "WTO_ITA": dict(
        disaggregate=1,
        category="Clean disaggregation (no member rows)",
        reasoning=("Each EU member state is independently a WTO member and signatory "
                   "of the Information Technology Agreement (1996) and ITA II (2015). "
                   "Although the EU customs union applies the ITA tariff schedule "
                   "uniformly, the underlying GATT commitments are made by individual "
                   "members. The DTI dataset captures this only at EU level."),
        collisions="None — no member-state row captures this commitment.",
        action=ALL27,
    ),
}


def classify_eu_row(rec):
    """Classify an EU row by content. Returns a key into EU_DECISIONS, or
    None if the row is not an EU row (or doesn't match any rule).

    Identification uses Subpillar plus regulation-ID detection in
    Act/Practice (primarily) and Description (as fallback context).
    Robust to row reordering.
    """
    if rec.get("Country") != "EUROPEAN UNION":
        return None

    sub_l = str(rec.get("Subpillar", "") or "").strip().lower()
    act = str(rec.get("Act_Practice", "") or "")
    cov = str(rec.get("Coverage", "") or "")
    desc = str(rec.get("Description", "") or "")
    act_l = act.lower()
    cov_l = cov.lower()
    desc_l = desc.lower()
    blob_l = act_l + " || " + cov_l + " || " + desc_l

    # ----- Tariff stat rows -----
    if "effective tariff rate on ict goods" in sub_l:
        return "TARIFF_RATE"
    if "coverage rate of zero-tariffs" in sub_l:
        return "TARIFF_ZERO"

    # ----- Local presence requirements (3 EU rows: GDPR, DSA, NIS) -----
    if "local presence requirements for digital services" in sub_l:
        if "2016/679" in blob_l or "general data protection regulation" in blob_l:
            return "GDPR_LOCAL_PRESENCE"
        if "2022/2065" in act_l or "digital services act" in act_l:
            return "DSA_LOCAL_PRESENCE"
        if "2016/1148" in act_l:
            return "NIS_LOCAL_PRESENCE"

    # ----- Single-row subpillars -----
    if "de minimis" in sub_l:
        return "DE_MINIMIS"
    if "blocking or filtering" in sub_l:
        return "RUSSIA_BLOCKING"
    if "open and transparent standard-setting" in sub_l:
        return "NLF_STANDARD_SETTING"
    if "self-certification for product safety" in sub_l:
        return "SELF_CERT"
    if "wto telecom reference paper" in sub_l:
        return "WTO_TELECOM"
    if "ban to transfer and local processing" in sub_l:
        return "PNR"
    if "trade agreements committing to open" in sub_l:
        return "EU_UK_TCA"
    if "exclusion from public procurement" in sub_l:
        return "IPI"
    if "wto agreement on government procurement" in sub_l:
        return "WTO_GPA"
    if "wto information technology agreement" in sub_l:
        return "WTO_ITA"

    # ----- Export restrictions (3 EU rows distinguished by Act/Practice) -----
    # Description text alone is unreliable: the Russia row mentions 2021/821
    # in passing, which would otherwise mis-route it to dual-use.
    if "export restrictions on ict goods" in sub_l:
        if "2021/821" in act_l:
            return "DUAL_USE_EXPORT"
        if "2023/1529" in act_l:
            return "BELARUS_EXPORT"
        if "833/2014" in act_l:
            return "RUSSIA_EXPORT"

    # ----- Conditional flow regime (2 EU rows: GDPR alone, GDPR+Data Act) -----
    if "conditional flow regime" in sub_l:
        if "2023/2854" in blob_l:
            return "DATA_ACT"
        if "2016/679" in blob_l or "general data protection regulation" in blob_l:
            return "GDPR_CONDITIONAL"

    # ----- DPIA/DPO requirement -----
    if "data protection impact assessment" in sub_l:
        return "GDPR_DPIA"

    # ----- Mandatory disclosure (3 EU rows: Trade Secrets Dir, DSA, P2B) -----
    if "mandatory disclosure of business trade secrets" in sub_l:
        if "2016/943" in act_l:
            return "TRADE_SECRETS_DIR"
        if "2022/2065" in act_l or "digital services act" in act_l:
            return "DSA_DISCLOSURE"
        if "2019/1150" in act_l:
            return "P2B"

    # ----- Antidumping (5 EU rows distinguished by Act/Practice) -----
    if "antidumping" in sub_l:
        if "2015/1953" in act_l:
            return "ANTIDUMPING_2015"
        if "2024/3014" in act_l:
            return "ANTIDUMPING_2024"
        if "2021/2011" in act_l:
            return "ANTIDUMPING_2021_2011"
        if "2022/72" in act_l:
            return "ANTIDUMPING_2022_72"
        # Silicon metal row (Act = "Antidumping measure", Coverage = silicon)
        if "silicon" in cov_l or "silicon" in desc_l:
            return "ANTIDUMPING_SILICON"

    return None


# Apply classification: tag every record with Disaggregate, build EU reasoning list
eu_reasoning_records = []
eu_row_counter = 0

for idx, rec in enumerate(results):
    key = classify_eu_row(rec)
    if key is None:
        rec["Disaggregate"] = 0
        if rec.get("Country") == "EUROPEAN UNION":
            print(f"WARNING: unclassified EU row at index {idx}: "
                  f"{str(rec.get('Subpillar',''))[:60]} | "
                  f"{str(rec.get('Act_Practice',''))[:60]}")
        continue

    decision = EU_DECISIONS[key]
    rec["Disaggregate"] = decision["disaggregate"]
    eu_row_counter += 1

    desc_full = str(rec.get("Description", "") or "")
    desc_snip = (desc_full[:300] + "...") if len(desc_full) > 300 else desc_full

    eu_reasoning_records.append({
        "EU_Row_Number": eu_row_counter,
        "Excel_Row_In_Main_Sheet": idx + 2,  # +2 = 1-indexing + header row
        "Subpillar": rec.get("Subpillar", ""),
        "Subpillar_ID": rec.get("Subpillar_ID", ""),
        "Act_Practice": rec.get("Act_Practice", ""),
        "Coverage": rec.get("Coverage", ""),
        "Description_Snippet": desc_snip,
        "Disaggregate": decision["disaggregate"],
        "Decision_Category": decision["category"],
        "Reasoning": decision["reasoning"],
        "Member_State_Collisions": decision["collisions"],
        "Recommended_Action_If_Disaggregating": decision["action"],
    })

# Sanity check
n_disagg_1 = sum(1 for r in results if r.get("Disaggregate") == 1)
n_disagg_0 = sum(1 for r in results if r.get("Disaggregate") == 0)
print(f"EU reasoning records: {len(eu_reasoning_records)} (expected 29)")
print(f"Disaggregate=1: {n_disagg_1} (expected 22)")
print(f"Disaggregate=0: {n_disagg_0} (expected {len(results)-22})")

## 14 · Write Output Excel

In [ ]:
wb = Workbook()

ID_COLS = [
    "Country", "Pillar", "Subpillar", "Subpillar_ID",
    "Act_Practice", "Coverage", "Description", "Timeframe",
    "Enabling_or_Restriction", "Partner_Specific",
]
DATE_COLS = [
    "Date_Entry_Into_Force", "Date_Enacted_First_Reported",
    "Date_Last_Amended", "Expiry_End_Date",
    "Date_Last_Extension", "Other_Date",
]

# Build Sheet 1 columns: ID + dates + per-category effect/direction +
# Disaggregate at the end. The aggregate Telecom_computer_info_services
# columns are replaced in-place by the six 9.1 / 9.2 / 9.3 columns.
s1_cols = ID_COLS + DATE_COLS
for cat in BPM6_CATS:
    if cat == "Telecom_computer_info_services":
        s1_cols += [
            "Telecom_9_1_effect", "Telecom_9_1_direction",
            "Telecom_9_2_effect", "Telecom_9_2_direction",
            "Telecom_9_3_effect", "Telecom_9_3_direction",
        ]
    else:
        s1_cols += [f"{cat}_effect", f"{cat}_direction"]
s1_cols += ["Disaggregate"]

# Build Sheet 2 columns: ID + per-category reasoning.
ID_COLS_REASONING = [c for c in ID_COLS if c != "Partner_Specific"]
s2_cols = list(ID_COLS_REASONING)
for cat in BPM6_CATS:
    if cat == "Telecom_computer_info_services":
        s2_cols.append("Telecom_9_1_9_2_9_3_reasoning")
    else:
        s2_cols.append(f"{cat}_reasoning")
s2_cols.append("Partner_Specific_reasoning")


def _write(ws, cols, tag):
    """Write a main sheet with header styling and column widths."""
    for ci, c in enumerate(cols, 1):
        cell = ws.cell(row=1, column=ci, value=c)
        cell.font = Font(bold=True, size=9, name="Arial")
        cell.alignment = Alignment(wrap_text=True, horizontal="center")
    for ri, rec in enumerate(results, 2):
        if ri % 2000 == 0:
            print(f"  {tag} row {ri:,}")
        for ci, c in enumerate(cols, 1):
            v = rec.get(c, "")
            if pd.isna(v):
                v = ""
            ws.cell(row=ri, column=ci, value=v)
    for i in range(1, len(cols) + 1):
        ws.column_dimensions[get_column_letter(i)].width = 14


def _write_eu_reasoning(ws, records):
    """Write the EU_Disaggregation_Reasoning sheet with no styling.

    Plain values only — no fonts, fills, borders, alignment, or column
    widths. This avoids the 'Excel found a problem with content' warning
    that arises when style references are introduced and later mutated.
    """
    headers = [
        "EU_Row_Number",
        "Excel_Row_In_Main_Sheet",
        "Subpillar",
        "Subpillar_ID",
        "Act_Practice",
        "Coverage",
        "Description_Snippet",
        "Disaggregate",
        "Decision_Category",
        "Reasoning",
        "Member_State_Collisions",
        "Recommended_Action_If_Disaggregating",
    ]
    for ci, h in enumerate(headers, 1):
        ws.cell(row=1, column=ci, value=h)
    for ri, rec in enumerate(records, 2):
        for ci, h in enumerate(headers, 1):
            v = rec.get(h, "")
            if pd.isna(v):
                v = ""
            ws.cell(row=ri, column=ci, value=v)


ws1 = wb.active
ws1.title = "Regulation Mapping"
_write(ws1, s1_cols, "S1")

ws2 = wb.create_sheet("Reasoning")
_write(ws2, s2_cols, "S2")

ws3 = wb.create_sheet("EU_Disaggregation_Reasoning")
_write_eu_reasoning(ws3, eu_reasoning_records)

wb.save(OUTPUT_PATH)

import os
print(f"Saved: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH) / 1024 / 1024:.1f} MB)")
print(f"  Sheet 1 (Regulation Mapping):           {len(s1_cols)} columns x {len(results):,} rows")
print(f"  Sheet 2 (Reasoning):                    {len(s2_cols)} columns x {len(results):,} rows")
print(f"  Sheet 3 (EU_Disaggregation_Reasoning):  12 columns x {len(eu_reasoning_records)} rows")

## 15 · Verification

In [ ]:
v1 = pd.read_excel(OUTPUT_PATH, sheet_name="Regulation Mapping")
v2 = pd.read_excel(OUTPUT_PATH, sheet_name="Reasoning")
v3 = pd.read_excel(OUTPUT_PATH, sheet_name="EU_Disaggregation_Reasoning")

assert v1.shape == (6992, 51), f"Sheet 1: {v1.shape}"
assert v2.shape == (6992, 25), f"Sheet 2: {v2.shape}"
assert v3.shape == (29, 12),   f"Sheet 3: {v3.shape}"

# Column-name sanity checks
assert "Telecom_9_1_effect"    in v1.columns
assert "Telecom_9_2_effect"    in v1.columns
assert "Telecom_9_3_effect"    in v1.columns
assert "Telecom_9_1_direction" in v1.columns
assert "Telecom_9_2_direction" in v1.columns
assert "Telecom_9_3_direction" in v1.columns
assert "Partner_Specific"      in v1.columns
assert "Disaggregate"          in v1.columns
assert v1.columns[-1] == "Disaggregate", "Disaggregate must be the last column"
assert "Telecom_computer_info_services_effect"    not in v1.columns
assert "Telecom_computer_info_services_direction" not in v1.columns
assert "Telecom_9_1_9_2_9_3_reasoning"            in v2.columns
assert "Partner_Specific_reasoning"               in v2.columns

# Direction integrity: effect=0 must imply direction=0
for sub in ("Telecom_9_1", "Telecom_9_2", "Telecom_9_3"):
    bad = v1[(v1[f"{sub}_effect"] == 0) & (v1[f"{sub}_direction"] != 0)]
    assert len(bad) == 0, f"{sub}: {len(bad)} rows with direction != 0 when effect = 0"

# Disaggregate flag integrity:
#   - Only EU rows can have Disaggregate=1
#   - Exactly 22 rows should have Disaggregate=1
non_eu_with_flag = v1[(v1["Country"] != "EUROPEAN UNION") & (v1["Disaggregate"] == 1)]
assert len(non_eu_with_flag) == 0, (
    f"Non-EU rows with Disaggregate=1: {len(non_eu_with_flag)} (expected 0)"
)
n_disagg = int((v1["Disaggregate"] == 1).sum())
assert n_disagg == 22, f"Disaggregate=1 count: {n_disagg} (expected 22)"

# EU reasoning sheet integrity
assert "Disaggregate" in v3.columns
assert "Recommended_Action_If_Disaggregating" in v3.columns
n_v3_disagg = int((v3["Disaggregate"] == 1).sum())
assert n_v3_disagg == 22, f"EU reasoning Disaggregate=1: {n_v3_disagg} (expected 22)"

print(f"Regulation Mapping:           {v1.shape[0]:,} rows x {v1.shape[1]} cols  OK")
print(f"Reasoning:                    {v2.shape[0]:,} rows x {v2.shape[1]} cols  OK")
print(f"EU_Disaggregation_Reasoning:  {v3.shape[0]:,} rows x {v3.shape[1]} cols  OK")
print(f"Telecom 9.1: {(v1['Telecom_9_1_effect']==1).sum():,} affected rows")
print(f"Telecom 9.2: {(v1['Telecom_9_2_effect']==1).sum():,} affected rows")
print(f"Telecom 9.3: {(v1['Telecom_9_3_effect']==1).sum():,} affected rows")
print(f"Partner_Specific: {(v1['Partner_Specific']==1).sum():,} flagged rows")
print(f"Disaggregate=1:   {n_disagg:,} EU rows flagged for member-state disaggregation")

---

## Appendix · Reference Material Summary

### Technical Annex (DTI)
Defines 12 pillars / 65 indicators with R (Restrictive) and E (Enabling) criteria.
The `Enabling or Restriction` column directly applies these criteria.

### BPM6 Chapter 10
Defines 15 goods/services categories in Table 10.1.
Key category for digital trade: **Telecom, Computer & Information Services**
(§§10.141–10.146), which is split in this notebook into three subcomponents:

* **9.1 Telecommunications** (§§10.141–10.142) — transmission services
* **9.2 Computer services** (§§10.143–10.145) — software, hosting, data processing
* **9.3 Information services** (§10.146) — news agencies, databases, search portals

### Partner_Specific column
Flags regulations whose treatment depends on the partner country (reciprocity,
sanctions, regional-bloc preferences, bilateral carve-outs, GPA-only
procurement eligibility, etc.). Pure descriptive *"Country X is/is not a
signatory"* entries are filtered out.